# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import itertools
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load Dataset

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)
print(y.shape)

(569, 30)
(569,)


# Split Dataset

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Hyperparameter Grid

In [4]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

# Run Hyperparameter Experiments

In [5]:
results = []

for n_estimators, max_depth, min_samples_split in itertools.product(
        param_grid['n_estimators'],
        param_grid['max_depth'],
        param_grid['min_samples_split']):

    start = time.time()

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    cv_score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='accuracy'
    ).mean()

    model.fit(X_train, y_train)

    train_acc = accuracy_score(
        y_train,
        model.predict(X_train)
    )

    test_acc = accuracy_score(
        y_test,
        model.predict(X_test)
    )

    training_time = time.time() - start

    gap = train_acc - test_acc

    results.append([
        n_estimators,
        max_depth,
        min_samples_split,
        cv_score,
        train_acc,
        test_acc,
        gap,
        training_time
    ])

# Results DataFrame

In [6]:
columns = [
    'n_estimators',
    'max_depth',
    'min_samples_split',
    'cv_accuracy',
    'train_accuracy',
    'test_accuracy',
    'overfit_gap',
    'training_time'
]

results_df = pd.DataFrame(
    results,
    columns=columns
)

results_df

,n_estimators,max_depth,min_samples_split,cv_accuracy,train_accuracy,test_accuracy,overfit_gap,training_time
0,50,3,2,0.951648,0.978022,0.947368,0.030654,0.852613
1,50,3,5,0.951648,0.975824,0.947368,0.028456,1.061637
2,50,5,2,0.951648,0.993407,0.956140,0.037266,1.175112
3,50,5,5,0.949451,0.993407,0.956140,0.037266,1.213922
4,50,10,2,0.951648,1.000000,0.956140,0.043860,1.213483
5,50,10,5,0.958242,0.993407,0.947368,0.046038,1.368840
6,100,3,2,0.951648,0.980220,0.956140,0.024079,2.504371
7,100,3,5,0.953846,0.978022,0.956140,0.021882,0.867418
8,100,5,2,0.949451,0.993407,0.956140,0.037266,0.964247
9,100,5,5,0.951648,0.993407,0.956140,0.037266,0.964022


# Sort by Best Accuracy

In [7]:
results_df.sort_values(
    by='cv_accuracy',
    ascending=False,
    inplace=True
)

results_df.head(10)

,n_estimators,max_depth,min_samples_split,cv_accuracy,train_accuracy,test_accuracy,overfit_gap,training_time
16,200,10,2,0.960440,1.000000,0.956140,0.043860,3.645381
5,50,10,5,0.958242,0.993407,0.947368,0.046038,1.368840
11,100,10,5,0.953846,0.995604,0.947368,0.048236,0.980814
7,100,3,5,0.953846,0.978022,0.956140,0.021882,0.867418
10,100,10,2,0.953846,1.000000,0.956140,0.043860,0.979933
17,200,10,5,0.953846,0.995604,0.947368,0.048236,2.338036
15,200,5,5,0.951648,0.993407,0.956140,0.037266,1.886600
13,200,3,5,0.951648,0.984615,0.956140,0.028475,3.004896
4,50,10,2,0.951648,1.000000,0.956140,0.043860,1.213483
2,50,5,2,0.951648,0.993407,0.956140,0.037266,1.175112


# result save in CSV

In [8]:
results_df.to_csv(
    "hyperparameter_results.csv",
    index=False
)

# LLM PART

In [9]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


In [10]:
import pandas as pd
from groq import Groq

In [11]:
import os
from google.colab import userdata
gem_key = userdata.get('groq')
os.environ['GROQ_API_KEY'] = gem_key

In [12]:
client = Groq()
client

In [22]:
results_text = results_df.to_string(index=False)

In [13]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello! How can I help you today?


# prompt_template(tot)

In [23]:
prompt_template = f"""
You are an expert Machine Learning Engineer.

Use Tree-of-Thought reasoning.

ROOT QUESTION:
Which hyperparameter configuration is best?

BRANCH A: Performance Analysis
- Identify configurations with highest CV accuracy.
- Rank top candidates.

BRANCH B: Overfitting Analysis
- Analyze train_accuracy vs test_accuracy.
- Use overfit_gap.
- Identify risky configurations.

BRANCH C: Training Cost Analysis
- Compare training_time.
- Determine if extra time provides meaningful gains.

BRANCH D: Bias-Variance Tradeoff
- Analyze max_depth values.
- Identify underfitting and overfitting cases.
- Determine balanced configurations.

MERGE STEP:
- Combine findings from all branches.
- Eliminate weak candidates.

FINAL DECISION:
- Select ONE best configuration.
- Explain why it is superior.

Hyperparameter Results:

{results_text}
"""

# Send Prompt to LLM

In [24]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt_template
        }
    ]
)

analysis = response.choices[0].message.content

print(analysis)

**Tree‑of‑Thought Reasoning**

Below each branch is explored separately, then the insights are merged to pick a single “best” configuration.

---

## 🌳 Branch A – Performance Analysis  

| n_estimators | max_depth | min_samples_split | **cv_accuracy** | train | test | overfit_gap | time (s) |
|--------------|-----------|-------------------|----------------|-------|------|-------------|----------|
| **200** | **10** | **2** | **0.96044** | 1.000 | 0.956 | 0.0439 | 3.65 |
| 50 | 10 | 5 | 0.95824 | 0.993 | 0.947 | 0.0460 | 1.37 |
| 100 | 10 | 2 | 0.95385 | 1.000 | 0.956 | 0.0439 | 0.98 |
| 100 | 3 | 5 | 0.95385 | 0.978 | 0.956 | **0.0219** | **0.87** |
| … (remaining rows) | … | … | ≤ 0.95385 | … | … | … | … |

* **Highest CV accuracy** – 0.96044 (200 / 10 / 2).  
* **Second‑best** – 0.95824 (50 / 10 / 5).  
* Many configurations cluster around 0.951‑0.954 CV, but they often enjoy lower training cost and smaller over‑fit gaps.

---

## 🌳 Branch B – Over‑fitting Analysis  

`overfit_gap = 

# Save LLM Analysis

In [25]:
with open("tree_of_thought_analysis.txt", "w") as f:
    f.write(analysis)

print("Analysis saved successfully")

Analysis saved successfully


# Extract Best Configuration Programmatically

In [26]:
best_config = results_df.iloc[0]

print(best_config)

n_estimators         200.000000
max_depth             10.000000
min_samples_split      2.000000
cv_accuracy            0.960440
train_accuracy         1.000000
test_accuracy          0.956140
overfit_gap            0.043860
training_time          3.645381
Name: 16, dtype: float64


# Save to csv

In [21]:
results_df.to_csv(
    "hyperparameter_results.csv",
    index=False
)

with open("tree_of_thought_analysis.txt", "w") as f:
    f.write(analysis)